In [ ]:
FIGURE = 'Fig.S9'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm, LinearSegmentedColormap
import matplotlib.ticker as ticker
import os
shared_legend_dir = str(output_path('Fig.S9/shared_legend'))
os.makedirs(shared_legend_dir, exist_ok=True)
time_scale_ranges = ['1', '2-3', '4-6', '7-12', '13-18', '19-24']
seasons = ['Spring']
season_labels = {'Spring': 'MAM', 'Summer': 'JJA', 'Autumn': 'SON', 'Winter': 'DJF'}
n_windows = 11
TARGET_W_IN = 8.0
SAVE_DPI = 600
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'stixsans'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
bg = '#FFFFFF'
nodata_bg = '#EEF3F8'
spine_col = '#000000'
txt_col = '#000000'
colors_list = ['#DCEAF6', '#C4DAEE', '#A7C7E3', '#84AFD4', '#5F93C2', '#3E78AE', '#255E97', '#13467C', '#083061']
heat_cmap = LinearSegmentedColormap.from_list('NatureCoolBlues_Bold', colors_list, N=256)
heat_cmap.set_bad(color=nodata_bg)
vmin = 100.0
vmax = 1500.0
norm = LogNorm(vmin=vmin, vmax=vmax)
print(f'[shared_normalization]LogNorm vmin={vmin}, vmax={vmax}')
from pathlib import Path

def draw_heatmap(process_data_dir, output_dir, out_name, global_max_clusters_override=None):
    os.makedirs(output_dir, exist_ok=True)
    if global_max_clusters_override is not None:
        global_max_clusters = global_max_clusters_override
    else:
        global_max_clusters = 1
        for ts in time_scale_ranges:
            for s in seasons:
                p = os.path.join(process_data_dir, f'{ts}_{s}', f'heatmap_matrix_{ts}_{s}.csv')
                if os.path.exists(p):
                    df = pd.read_csv(p, index_col=0, encoding='utf-8-sig')
                    global_max_clusters = max(global_max_clusters, len(df))
    print(f'\n[{out_name}] global_max_clusters={global_max_clusters}')
    n_rows, n_cols = (len(seasons), len(time_scale_ranges))
    cell_w, cell_h = (0.1, 0.06)
    subplot_w = n_windows * cell_w
    subplot_h = global_max_clusters * cell_h
    gap_w, gap_h = (0.15, 0.2)
    margin_l, margin_r = (0.8, 0.6)
    margin_t, margin_b = (0.45, 0.65)
    cb_gap, cb_w_in = (0.1, 0.12)
    natural_w = margin_l + n_cols * subplot_w + (n_cols - 1) * gap_w + margin_r + cb_gap + cb_w_in
    natural_h = margin_t + n_rows * subplot_h + (n_rows - 1) * gap_h + margin_b
    s = TARGET_W_IN / natural_w

    def S(x):
        return x * s
    sw_s = n_windows * S(cell_w)
    sh_s = global_max_clusters * S(cell_h)
    gw_s, gh_s = (S(gap_w), S(gap_h))
    ml_s, mb_s = (S(margin_l), S(margin_b))
    fig_w = TARGET_W_IN
    fig_h = natural_h * s
    print(f'  canvas: {fig_w:.2f}×{fig_h:.2f} in')

    def to_norm_x(ci):
        return ((ml_s + ci * (sw_s + gw_s)) / fig_w, sw_s / fig_w)

    def to_norm_y(ri):
        return ((mb_s + (n_rows - 1 - ri) * (sh_s + gh_s)) / fig_h, sh_s / fig_h)
    all_cols = [f'{y}-{y + 9}' for y in range(2001, 2012)]
    full_lbls = [f'{y}-{y + 9}' for y in range(2001, 2012)]
    sparse_lbls = [full_lbls[i] if i % 2 == 0 else '' for i in range(n_windows)]
    x_edges = np.arange(0, n_windows + 1)
    y_edges = np.arange(0, global_max_clusters + 1)
    SPINE_W = 1.0
    fig = plt.figure(figsize=(fig_w, fig_h), facecolor=bg)
    for row_i, season in enumerate(seasons):
        for col_i, ts in enumerate(time_scale_ranges):
            xl, aw = to_norm_x(col_i)
            yb, ah = to_norm_y(row_i)
            ax = fig.add_axes([xl, yb, aw, ah])
            ax.set_facecolor(nodata_bg)
            for sp in ax.spines.values():
                sp.set_color(spine_col)
                sp.set_linewidth(SPINE_W)
                sp.set_visible(True)
            hm_path = os.path.join(process_data_dir, f'{ts}_{season}', f'heatmap_matrix_{ts}_{season}.csv')
            ax.set_xlim(0, n_windows)
            ax.set_ylim(global_max_clusters, 0)
            if row_i == n_rows - 1:
                ax.set_xticks(np.arange(n_windows) + 0.5)
                ax.set_xticklabels(sparse_lbls, fontsize=10, rotation=90, ha='center', va='top', color=txt_col)
                ax.tick_params(axis='x', length=3, width=0.8, pad=2, colors=txt_col, bottom=True, top=False, labelbottom=True)
            else:
                ax.set_xticks([])
                ax.tick_params(axis='x', bottom=True, top=False, labelbottom=False)
            ax.set_yticks([])
            ax.tick_params(axis='y', left=True, right=False, labelleft=False)
            if row_i == 0:
                ax.set_title(f'{ts}', fontsize=15, pad=10, color=txt_col)
            if col_i == 0:
                ax.set_ylabel(season_labels[season], fontfamily='Times New Roman', fontsize=20, labelpad=4, color=txt_col)
            if not os.path.exists(hm_path):
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=15, color='#CCCCCC', style='italic')
                continue
            df_hm = pd.read_csv(hm_path, index_col=0, encoding='utf-8-sig')
            n_cl = len(df_hm)
            if n_cl < global_max_clusters:
                pad_df = pd.DataFrame(np.zeros((global_max_clusters - n_cl, n_windows)), columns=df_hm.columns, index=[f'PAD_{i}' for i in range(global_max_clusters - n_cl)])
                df_hm = pd.concat([df_hm, pad_df])
            for c in all_cols:
                if c not in df_hm.columns:
                    df_hm[c] = 0
            df_hm = df_hm[all_cols]
            mat = df_hm.values.astype(float)
            ax.pcolormesh(x_edges, y_edges, np.ma.masked_where(mat == 0, mat), cmap=heat_cmap, norm=norm, edgecolors='none', linewidth=0, antialiased=False, shading='flat')
            ax.text(0.15, 0.05, f'n={n_cl}', transform=ax.transAxes, fontsize=15, color=txt_col, ha='left', va='bottom', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#CCCCCC', linewidth=0.5, alpha=0.9))
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(output_dir, f'{out_name}.{ext}'), dpi=SAVE_DPI, facecolor=bg, bbox_inches='tight')
    plt.show()
    print(f'  ✓ saved: {out_name}.png / .pdf  →  {output_dir}')

def main():
    import pandas as pd
    folder = INPUT_DIR
    pathways = ['SPEI', 'SSMIrz']
    scales = ['1', '2-3', '4-6', '7-12', '13-18', '19-24']
    seasons = ['Spring']
    rows = [len(pd.read_csv(folder / p / f'{scale}_{season}' / f'heatmap_matrix_{scale}_{season}.csv', index_col=0)) for p in pathways for scale in scales for season in seasons]
    for pathway in pathways:
        draw_heatmap(str(folder / pathway), str(OUTPUT / 'Fig.S9' / pathway), f'gantt_{pathway}_600dpi', global_max_clusters_override=max(rows))
if __name__ == '__main__':
    main()
